In [19]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated , Literal
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field 
import operator

In [20]:
from typing import TypedDict, Literal, Dict

class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive', 'negative']
    diagnosis: Dict
    response: str

In [21]:
model = ChatOllama(model='Phi3')

In [22]:
class SentimentSchema(BaseModel): 
    sentiment : Literal ['positive', 'negative'] = Field (description='Sentiment of the review')

In [23]:
class DiagnosisSchema(BaseModel) : 

    issue_type : Literal["UX" , "Performence","Bug" , "Support" , "Other"] = Field (description='the category of issues mentioned in the review')
    tone : Literal["angry","frustated","disappointed","calm"] = Field (description='the emotional tone expressed by user')
    urgency : Literal["low","medium","high"] = Field(description='How urgent or critical the issues to be')


In [24]:
structured_model = model.with_structured_output(SentimentSchema)
structured_model2 = model.with_structured_output(DiagnosisSchema)



In [25]:
prompt = 'What is the sentiment of the following review - The  software too good  '
structured_model.invoke(prompt).sentiment

'positive'

In [26]:
def positive_response(state: ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:

{state["review"]}

Also kindly ask the user to leave feedback on our website."""
    
    response = model.invoke(prompt).content
    return {"response": response}


def run_diagnosis(state: ReviewState):
    prompt = f"""Diagnose this negative review:

{state['review']}

Return issue_type, tone, and urgency."""
    
    response = structured_model2.invoke(prompt)
    return {"diagnosis": response.model_dump()}


def negative_response(state: ReviewState):
    diagnosis = state["diagnosis"]

    prompt = f"""You are a support assistant. The user had a "{diagnosis['issue_type']}" issue,
sounded "{diagnosis['tone']}", and marked urgency.
Write an empathetic and helpful resolution message."""
    
    response = model.invoke(prompt).content
    return {"response": response}





In [28]:
graph = StateGraph(ReviewState)

# Nodes
graph.add_node("find_sentiment", find_sentiment)
graph.add_node("positive_response", positive_response)
graph.add_node("run_diagnosis", run_diagnosis)
graph.add_node("negative_response", negative_response)

# Start
graph.add_edge(START, "find_sentiment")

# Conditional flow
graph.add_conditional_edges(
    "find_sentiment",
    check_sentiment,
    {
        "positive_response": "positive_response",
        "run_diagnosis": "run_diagnosis",
    },
)

# Negative path
graph.add_edge("run_diagnosis", "negative_response")

# End
graph.add_edge("positive_response", END)
graph.add_edge("negative_response", END)

workflow = graph.compile()




input_data = {
    "review": "The app is very slow and keeps crashing"
}

result = workflow.invoke(input_data)
print(result)


{'review': 'The app is very slow and keeps crashing', 'sentiment': 'negative', 'diagnosis': {'issue_type': 'Performence', 'tone': 'frustated', 'urgency': 'high'}, 'response': "Hello! I'm truly sorry to hear about the performance issue you’re experiencing with our service – it sounds incredibly frustrating, especially given your sense of urgency regarding this matter. Your satisfaction is very important to us, and we understand how critical uninterrupted access can be for your daily work or personal activities.\n\nI assure you that I am here to assist in resolving the issue as swiftly as possible. To provide immediate support, please let me know a brief description of the problem so I can begin troubleshooting steps with more specificity and potentially escalate this matter if required. We aim to get back on track promptly because we appreciate your patience during this time.\n\nThank you for bringing this issue to our attention – together, we will work towards a resolution that restore

In [ ]:
workflow

In [29]:
input_data = {
    "review": "The app is very slow and keeps crashing"
}

result = workflow.invoke(input_data)
print(result)

{'review': 'The app is very slow and keeps crashing', 'sentiment': 'negative', 'diagnosis': {'issue_type': 'Performence', 'tone': 'frustated', 'urgency': 'high'}, 'response': "I'm really sorry to hear that you're experiencing frustration due to performance issues on our platform. It is indeed important as it impacts your user experience, which we aim to maintain at the highest possible standard. I understand how crucial this matter seems for you and hence assure you that we will act urgently towards finding a solution.\n\nWe're already escalating your issue within our technical team who are working relentlessly on identifying & fixing these problems promptly. In parallel, if there is any specific way or functioning feature of the platform which helps ease user experience for you and works without lags consistently, we would appreciate it to know about that as well. This could assist us in maintaining high-performance aspects while resolving this issue.\n\nI'm here with regular updates 